# 04 — Two-Tower Test Evaluation

Loads a trained checkpoint and evaluates it on the test split using the same
ranking metrics as the ALS baseline:
- **Precision@K** — fraction of top-K recommendations that are in the test ground truth
- **Recall@K** — fraction of test ground-truth items that appear in top-K
- **NDCG@K** — graded relevance using `r = 1 + 2*is_read + beta*max(0, rating-3)`

Evaluated at K ∈ {10, 50, 100}.

Run from the `two_tower/` directory.

In [1]:
import sys
from pathlib import Path

# Resolve to two_tower/ so src/ imports work
ROOT = Path("__file__").resolve().parents[1]
sys.path.insert(0, str(ROOT))

In [2]:
import math
import time
from collections import defaultdict
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import torch
import fsspec
import pyarrow.parquet as pq
from omegaconf import OmegaConf
from tqdm.auto import tqdm

from src.data.loaders import load_artifacts, build_item_feature_tensors, _iter_parquet_batches
from src.models.two_tower import TwoTowerModel

/home/s38976581_gmail_com/micromamba/envs/fin_sentiment/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

In [3]:
cfg = OmegaConf.load("../configs/baseline.yaml")

# ── Edit these paths ──────────────────────────────────────────────────────────
CHECKPOINT_PATH = "../checkpoints/two_tower_baseline_2026-03-14_05-55-29/epoch_01_val6.9274.pt"

# MLflow run that produced the checkpoint — used to fetch vocab_and_stats.pt
MLFLOW_RUN_ID = "d37acf8b358a4e8b9609598ef511264b"

# Test split on GCS (same bucket layout as train/val)
TEST_PATH = "gs://nshen7-personal-bucket/projects/rec_sys_goodreads/data/two_tower_splits/test"

# beta used to compute interaction strength r (match ALS baseline)
BETA = 0.5

# K values to evaluate
K_VALUES = [10, 50, 100]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ENCODE_BATCH_SIZE = 2048   # items/users encoded per forward pass

# ANN index config — IVFFlat with nlist=4096 gives ~50x speedup vs exact search
# nprobe controls recall/speed trade-off at query time (higher = more accurate)
FAISS_NLIST  = 4096
FAISS_NPROBE = 64

# Cached item embeddings saved next to the checkpoint to avoid re-encoding
ITEM_VECS_CACHE = Path(CHECKPOINT_PATH).parent / "item_vecs.npy"

print(f"Checkpoint      : {CHECKPOINT_PATH}")
print(f"MLflow run      : {MLFLOW_RUN_ID}")
print(f"Test path       : {TEST_PATH}")
print(f"Device          : {DEVICE}")
print(f"Beta            : {BETA}")
print(f"K values        : {K_VALUES}")
print(f"Item vecs cache : {ITEM_VECS_CACHE}")

Checkpoint      : ../checkpoints/two_tower_baseline_2026-03-14_05-55-29/epoch_01_val6.9274.pt
MLflow run      : d37acf8b358a4e8b9609598ef511264b
Test path       : gs://nshen7-personal-bucket/projects/rec_sys_goodreads/data/two_tower_splits/test
Device          : cuda
Beta            : 0.5
K values        : [10, 50, 100]
Item vecs cache : ../checkpoints/two_tower_baseline_2026-03-14_05-55-29/item_vecs.npy


## 1. Load artifacts and model

In [4]:
print("Loading artifacts...")
artifacts = load_artifacts('../' + cfg.data.artifacts_path)
print(f"  num_users={artifacts['num_users']:,}  num_items={artifacts['num_items']:,}")

print("Building item feature tensors...")
_ITEM_FEAT_COLS = [
    "target_item_id",
    "book_primary_author_id", "book_language", "book_format", "book_top_shelves",
    "book_avg_rating", "book_ratings_count", "book_num_pages",
    "book_publication_year", "book_is_ebook",
]
item_cat_feats, item_num_feats = build_item_feature_tensors(
    _iter_parquet_batches(cfg.data.train_path, columns=_ITEM_FEAT_COLS),
    artifacts,
)
print(f"  item_cat_feats: {tuple(item_cat_feats.shape)}")
print(f"  item_num_feats: {tuple(item_num_feats.shape)}")

Loading artifacts...
  num_users=876,134  num_items=2,260,810
Building item feature tensors...


Building item feature tensors: 800it [03:34,  3.73it/s]

  item_cat_feats: (2260811, 6)
  item_num_feats: (2260811, 5)


In [5]:
print("Loading model checkpoint...")
model = TwoTowerModel(cfg, artifacts).to(DEVICE)
ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"  Loaded epoch {ckpt['epoch']}  (val_loss={ckpt['val_loss']:.4f})")

Loading model checkpoint...
  Loaded epoch 1  (val_loss=6.9274)


## 2. Load test split

In [6]:
# Columns needed for evaluation
_TEST_COLS = [
    "user_id", "target_item_id",
    "is_read", "rating",
    "history_item_ids", "history_item_weights",
]

print("Loading test split...")
test_batches = list(_iter_parquet_batches(TEST_PATH, columns=_TEST_COLS))
test_df = pd.concat(test_batches, ignore_index=True)
del test_batches

print(f"  {len(test_df):,} interactions")
print(f"  {test_df['user_id'].nunique():,} unique users")
print(f"  {test_df['target_item_id'].nunique():,} unique items")

Loading test split...
  4,230,805 interactions
  161,162 unique users
  528,878 unique items


## 3. Compute graded relevance

Using the same formula as the ALS baseline:
$$r = 1 + 2 \cdot \text{is\_read} + \beta \cdot \max(0,\; \text{rating} - 3)$$

In [7]:
test_df["r"] = (
    1
    + 2 * test_df["is_read"]
    + BETA * np.maximum(0, test_df["rating"] - 3)
)

# Ground-truth dict: user_id -> {item_id: r}
ground_truth: dict[int, dict[int, float]] = defaultdict(dict)
for row in test_df[["user_id", "target_item_id", "r"]].itertuples(index=False):
    ground_truth[row.user_id][row.target_item_id] = row.r

print(f"Ground truth built for {len(ground_truth):,} users")

Ground truth built for 161,162 users


## 4. Encode all items in the catalogue

In [8]:
if ITEM_VECS_CACHE.exists():
    print(f"Loading cached item embeddings from {ITEM_VECS_CACHE} ...")
    item_vecs = torch.from_numpy(np.load(ITEM_VECS_CACHE))
    print(f"  Item embedding matrix: {tuple(item_vecs.shape)}")
else:
    num_items = artifacts["num_items"]
    all_item_ids = torch.arange(1, num_items + 1, dtype=torch.long)

    item_vecs_list = []
    with torch.no_grad():
        for start in tqdm(range(0, len(all_item_ids), ENCODE_BATCH_SIZE), desc="Encoding items"):
            ids = all_item_ids[start : start + ENCODE_BATCH_SIZE]
            cat = item_cat_feats[ids].to(DEVICE)
            num = item_num_feats[ids].to(DEVICE)
            vecs = model.encode_items(ids.to(DEVICE), cat, num)   # [B, d_out]
            item_vecs_list.append(vecs.cpu())

    item_vecs = torch.cat(item_vecs_list, dim=0)   # [num_items, d_out]
    print(f"  Item embedding matrix: {tuple(item_vecs.shape)}")

    np.save(ITEM_VECS_CACHE, item_vecs.numpy())
    print(f"  Saved to {ITEM_VECS_CACHE}")

Loading cached item embeddings from ../checkpoints/two_tower_baseline_2026-03-14_05-55-29/item_vecs.npy ...


  Item embedding matrix: (2260810, 256)


In [9]:
# Build IVFFlat FAISS index (inner-product / cosine if embeddings are L2-normalised)
d = item_vecs.shape[1]
item_vecs_np = item_vecs.numpy().astype(np.float32)   # FAISS requires float32

quantizer = faiss.IndexFlatIP(d)
index = faiss.IndexIVFFlat(quantizer, d, FAISS_NLIST, faiss.METRIC_INNER_PRODUCT)

print(f"Training FAISS IVFFlat index (nlist={FAISS_NLIST}, d={d}, n={len(item_vecs_np):,})...")
index.train(item_vecs_np)
index.add(item_vecs_np)       # vectors added at positions 0..num_items-1  (item_id = pos+1)
index.nprobe = FAISS_NPROBE

print(f"Index ready: {index.ntotal:,} vectors, nprobe={FAISS_NPROBE}")

Training FAISS IVFFlat index (nlist=4096, d=256, n=2,260,810)...
Index ready: 2,260,810 vectors, nprobe=64


In [10]:
# One representative row per user (history is user-level, identical across rows)
user_rows = (
    test_df[["user_id", "history_item_ids", "history_item_weights"]]
    .drop_duplicates(subset=["user_id"])
    .reset_index(drop=True)
)
print(f"Encoding {len(user_rows):,} unique test users...")

max_k = max(K_VALUES)

# top_k_per_user[user_id] = list of top-max_k recommended item_ids (1-indexed)
top_k_per_user: dict[int, list[int]] = {}

with torch.no_grad():
    for start in tqdm(range(0, len(user_rows), ENCODE_BATCH_SIZE), desc="Encoding users"):
        batch_rows = user_rows.iloc[start : start + ENCODE_BATCH_SIZE]

        user_ids_t = torch.tensor(batch_rows["user_id"].values, dtype=torch.long).to(DEVICE)
        history_ids_t = torch.tensor(
            np.array(batch_rows["history_item_ids"].tolist(), dtype=np.int32),
            dtype=torch.long,
        ).to(DEVICE)
        history_wts_t = torch.tensor(
            np.array(batch_rows["history_item_weights"].tolist(), dtype=np.float32),
            dtype=torch.float32,
        ).to(DEVICE)

        user_vecs_batch = model.encode_users(user_ids_t, history_ids_t, history_wts_t)  # [B, d]
        user_vecs_np = user_vecs_batch.cpu().numpy().astype(np.float32)

        # ANN search: returns (scores [B, max_k], indices [B, max_k])
        # indices are 0-based positions in the FAISS index → item_id = idx + 1
        _, topk_indices = index.search(user_vecs_np, max_k)   # [B, max_k]

        for i, uid in enumerate(batch_rows["user_id"].values):
            top_k_per_user[int(uid)] = (topk_indices[i] + 1).tolist()

print(f"Recommendations generated for {len(top_k_per_user):,} users")

Encoding 161,162 unique test users...


Encoding users: 100%|██████████| 79/79 [01:18<00:00,  1.01it/s]

Recommendations generated for 161,162 users


In [11]:
### THE OLD APPROACH (exact search on GPU, much slower but simpler code)
# # One representative row per user (history is user-level, identical across rows)
# user_rows = (
#     test_df[["user_id", "history_item_ids", "history_item_weights"]]
#     .drop_duplicates(subset=["user_id"])
#     .reset_index(drop=True)
# )
# print(f"Encoding {len(user_rows):,} unique test users...")

# max_k = max(K_VALUES)

# # top_k_per_user[user_id] = list of top-max_k recommended item_ids (1-indexed)
# top_k_per_user: dict[int, list[int]] = {}

# item_vecs_device = item_vecs.to(DEVICE)  # keep on GPU for fast matmul

# with torch.no_grad():
#     for start in tqdm(range(0, len(user_rows), ENCODE_BATCH_SIZE), desc="Encoding users"):
#         batch_rows = user_rows.iloc[start : start + ENCODE_BATCH_SIZE]

#         user_ids_t = torch.tensor(batch_rows["user_id"].values, dtype=torch.long).to(DEVICE)
#         history_ids_t = torch.tensor(
#             np.array(batch_rows["history_item_ids"].tolist(), dtype=np.int32),
#             dtype=torch.long,
#         ).to(DEVICE)
#         history_wts_t = torch.tensor(
#             np.array(batch_rows["history_item_weights"].tolist(), dtype=np.float32),
#             dtype=torch.float32,
#         ).to(DEVICE)

#         user_vecs_batch = model.encode_users(user_ids_t, history_ids_t, history_wts_t)  # [B, d]

#         # scores[b, i] = dot(user_vec[b], item_vec[i])
#         scores = user_vecs_batch @ item_vecs_device.T   # [B, num_items]

#         # top-max_k item indices (0-indexed into item_vecs → item_id = idx+1)
#         topk_indices = torch.topk(scores, k=max_k, dim=1).indices.cpu().numpy()  # [B, max_k]

#         for i, uid in enumerate(batch_rows["user_id"].values):
#             top_k_per_user[int(uid)] = (topk_indices[i] + 1).tolist()  # convert to 1-indexed item_ids

# print(f"Recommendations generated for {len(top_k_per_user):,} users")

## 6. Compute ranking metrics

Identical formulas to `evaluate_ranking()` in `03_fit_als_full.py`.

In [12]:
def compute_ranking_metrics(
    top_k_per_user: dict[int, list[int]],
    ground_truth: dict[int, dict[int, float]],
    k: int,
) -> dict[str, float]:
    """
    Compute Precision@K, Recall@K, NDCG@K (graded relevance).

    Parameters
    ----------
    top_k_per_user : dict[user_id -> list of recommended item_ids (top-max_k)]
    ground_truth   : dict[user_id -> dict[item_id -> r]]
    k              : cutoff
    """
    eval_users = list(ground_truth.keys())
    n_users = len(eval_users)

    total_hits = 0
    sum_recall = 0.0
    sum_ndcg = 0.0

    for uid in eval_users:
        recs = top_k_per_user.get(uid, [])[:k]
        gt = ground_truth[uid]          # {item_id: r}

        n_relevant = len(gt)
        n_hits = sum(1 for item_id in recs if item_id in gt)

        # --- Precision@K ---
        total_hits += n_hits

        # --- Recall@K ---
        sum_recall += n_hits / max(n_relevant, 1)

        # --- NDCG@K (graded) ---
        # DCG
        dcg = sum(
            gt[item_id] / math.log2(rank + 2)  # rank is 0-indexed
            for rank, item_id in enumerate(recs)
            if item_id in gt
        )
        # IDCG: sort ground-truth by r desc, take top-k
        sorted_r = sorted(gt.values(), reverse=True)[:k]
        idcg = sum(r / math.log2(i + 2) for i, r in enumerate(sorted_r))
        sum_ndcg += dcg / idcg if idcg > 0 else 0.0

    return {
        f"precision@{k}": total_hits / (n_users * k),
        f"recall@{k}": sum_recall / n_users,
        f"ndcg@{k}": sum_ndcg / n_users,
    }

In [13]:
test_metrics_all = {}
for k in K_VALUES:
    t0 = time.time()
    metrics = compute_ranking_metrics(top_k_per_user, ground_truth, k=k)
    elapsed = time.time() - t0
    test_metrics_all[k] = metrics
    print(f"K={k:3d}  NDCG@K={metrics[f'ndcg@{k}']:.4f}  "
          f"P@K={metrics[f'precision@{k}']:.4f}  "
          f"R@K={metrics[f'recall@{k}']:.4f}  ({elapsed:.1f}s)")

K= 10  NDCG@K=0.0310  P@K=0.0270  R@K=0.0262  (1.4s)
K= 50  NDCG@K=0.0408  P@K=0.0136  R@K=0.0593  (2.1s)
K=100  NDCG@K=0.0469  P@K=0.0096  R@K=0.0791  (3.0s)


## 7. Summary

In [14]:
print("=" * 55)
print("  TWO-TOWER MODEL — TEST EVALUATION")
print("=" * 55)
print(f"Checkpoint : {CHECKPOINT_PATH}")
print(f"Test users : {len(ground_truth):,}")
print(f"Beta       : {BETA}  (r = 1 + 2*is_read + {BETA}*max(0, rating-3))")
print()
print(f"{'K':>5}  {'NDCG@K':>10}  {'Precision@K':>12}  {'Recall@K':>10}")
print("-" * 45)
for k in K_VALUES:
    m = test_metrics_all[k]
    print(f"{k:>5}  {m[f'ndcg@{k}']:>10.4f}  {m[f'precision@{k}']:>12.4f}  {m[f'recall@{k}']:>10.4f}")
print("=" * 55)

  TWO-TOWER MODEL — TEST EVALUATION
Checkpoint : ../checkpoints/two_tower_baseline_2026-03-14_05-55-29/epoch_01_val6.9274.pt
Test users : 161,162
Beta       : 0.5  (r = 1 + 2*is_read + 0.5*max(0, rating-3))

    K      NDCG@K   Precision@K    Recall@K
---------------------------------------------
   10      0.0310        0.0270      0.0262
   50      0.0408        0.0136      0.0593
  100      0.0469        0.0096      0.0791


## 8. (Optional) Per-user metric distribution

In [15]:
K_PLOT = 10  # which K to inspect

ndcg_per_user = []
recall_per_user = []

for uid, gt in ground_truth.items():
    recs = top_k_per_user.get(uid, [])[:K_PLOT]
    n_hits = sum(1 for item_id in recs if item_id in gt)
    recall_per_user.append(n_hits / max(len(gt), 1))

    dcg = sum(
        gt[item_id] / math.log2(rank + 2)
        for rank, item_id in enumerate(recs)
        if item_id in gt
    )
    sorted_r = sorted(gt.values(), reverse=True)[:K_PLOT]
    idcg = sum(r / math.log2(i + 2) for i, r in enumerate(sorted_r))
    ndcg_per_user.append(dcg / idcg if idcg > 0 else 0.0)

ndcg_arr = np.array(ndcg_per_user)
recall_arr = np.array(recall_per_user)

print(f"NDCG@{K_PLOT} distribution (n={len(ndcg_arr):,} users):")
for p in [0, 10, 25, 50, 75, 90, 100]:
    print(f"  p{p:3d}: {np.percentile(ndcg_arr, p):.4f}")
print()
print(f"Recall@{K_PLOT} distribution:")
for p in [0, 10, 25, 50, 75, 90, 100]:
    print(f"  p{p:3d}: {np.percentile(recall_arr, p):.4f}")

NDCG@10 distribution (n=161,162 users):
  p  0: 0.0000
  p 10: 0.0000
  p 25: 0.0000
  p 50: 0.0000
  p 75: 0.0000
  p 90: 0.0873
  p100: 1.0000

Recall@10 distribution:
  p  0: 0.0000
  p 10: 0.0000
  p 25: 0.0000
  p 50: 0.0000
  p 75: 0.0000
  p 90: 0.0556
  p100: 1.0000
